# Systemy uczące się - Zad. dom. 1: Minimalizacja ryzyka empirycznego
Celem zadania jest zaimplementowanie własnego drzewa decyzyjnego wykorzystującego idee minimalizacji ryzyka empirycznego. 

### Autor rozwiązania
Uzupełnij poniższe informacje umieszczając swoje imię i nazwisko oraz numer indeksu:

In [ ]:
NAME = "Jan Nowakowski"
ID = "155042"

## Twoja implementacja

Twoim celem jest uzupełnić poniższą klasę `TreeNode` tak by po wywołaniu `TreeNode.fit` tworzone było drzewo decyzyjne minimalizujące ryzyko empiryczne. Drzewo powinno wspierać problem klasyfikacji wieloklasowej (jak w przykładzie poniżej). Zaimplementowany algorytm nie musi (ale może) być analogiczny do zaprezentowanego na zajęciach algorytmu dla klasyfikacji. Wszelkie przejawy inwencji twórczej wskazane. **Pozostaw komenatrze w kodzie, które wyjaśniają Twoje rozwiązanie.**

Schemat oceniania:
- wynik na zbiorze Iris (automatyczna ewaluacja) celność klasyfikacji >= prostego baseline'u + 10%: +40%,
- wynik na ukrytym zbiorze testowym 1 (automatyczna ewaluacja) celność klasyfikacji >= prostego baseline'u + 15%: +30%,
- wynik na ukrytym zbiorze testowym 2 (automatyczna ewaluacja) celność klasyfikacji >= prostego baseline'u + 5%: +30%.

Niedozwolone jest korzystanie z zewnętrznych bibliotek do tworzenia drzewa decyzyjnego (np. scikit-learn). 
Możesz jedynie korzystać z biblioteki numpy.

#### Uwaga: Możesz dowolnie modyfikować elementy tego notebooka (wstawiać komórki i zmieniać kod), o ile będzie się w nim na koniec znajdowała kompletna implementacja klasy `TreeNode` w jednej komórce.

In [ ]:
import numpy as np

class TreeNode:
    def __init__(self, max_depth=5, min_samples_split=2, depth=0):
        self.left: TreeNode | None = None
        self.right: TreeNode | None = None
        
        self.feature = None                 # Cecha, po której następuje podział
        self.threshold = None               # Próg podziału dla tej cechy
        self.value = None                   # Wartość klasy jeżeli liść
        # Parametry drzewa
        self.max_depth = max_depth          # Maksymalna głębokość drzewa
        self.min_samples_split = min_samples_split   # Minimalna liczba próbek do podziału
        self.depth = depth                  # Aktualna warstwa drzewa
    #wyznaczenie entropii dla danego zbioru danych, w celu oceny jakości podziału
    def entropy(self, y: np.ndarray):
        counts = np.bincount(y) / len(y)     # Normalizacja do prawdopodobieństw
        prob = counts[counts > 0]            
        return -np.sum(prob * np.log2(prob)) 


    #funkcja dobierajaca threshldy i feature dla kolejnych nodeów drzewa na podstawie minimalizacji entropii,
    # oraz wywołująca rekurencyjnie kolejne node'y dla podzielonych danych, aż do osiągnięcia warunków zatrzymania (głębokość lub minimalna liczba próbek)
    def fit(self, data: np.ndarray, target: np.ndarray) -> None:
        # Sprawdzenie czy dany węzeł nie przekracza warunków głębokości lub minimalnej liczby elementów do podziału,
        # jeżeli tak zostaje liściem i klasa dominująca przypisana jako wartość
    
        if len(target) < self.min_samples_split or self.depth >= self.max_depth or len(np.unique(target)) == 1:
            self.value = np.bincount(target).argmax()
            return 
        
        best_entropy = float('inf')     
        best_split = None              
         # iteracja po wszystkich cechach oraz sprawdzanie kolejnych wartośći jako progów
        for feature in range(data.shape[1]):
            for threshold in np.unique(data[:, feature]):
                #przypisanie elementom strony podziału
                left_mask = data[:, feature] <= threshold
                right_mask = ~left_mask
                # sprawdzenie czy podział wprowadza jakiekolwiek zmiany
                if sum(left_mask) == 0 or sum(right_mask) == 0:
                    continue
                # wyznaczenie poziomu entropii dla podzbiorów
                entropy_left = self.entropy(target[left_mask])
                entropy_right = self.entropy(target[right_mask])
                #uśredniona i znormalizowana entropia dla całego podziału
                weighted_entropy = (sum(left_mask) * entropy_left + sum(right_mask) * entropy_right) / len(target)
                #sprawdzenie czy podział jest lepszy niż dotychczasowy najlepszy
                if weighted_entropy < best_entropy:
                    best_entropy = weighted_entropy
                    best_split = (feature, threshold, left_mask, right_mask)
        #utworzenie kolejnego poziomu drzewa dla podzielonych danych z głębokością zwiększoną o 1, i wywołanie treningu dla nowo utworzonych węzłów
        if best_split:
            self.feature, self.threshold, left_mask, right_mask = best_split
            self.left = TreeNode(max_depth=self.max_depth, min_samples_split=self.min_samples_split, depth=self.depth + 1)
            self.right = TreeNode(max_depth=self.max_depth, min_samples_split=self.min_samples_split, depth=self.depth + 1)
            self.left.fit(data[left_mask], target[left_mask])
            self.right.fit(data[right_mask], target[right_mask])
        else:
            # Jeśli nie znaleziono lepszego podziału, ten węzeł staje się liściem
            self.value = np.bincount(target).argmax()

    def predict(self, data: np.ndarray) -> np.ndarray:
        y_pred = np.zeros(data.shape[0])
        for i in range(data.shape[0]): #sprawdzanie każdego elementu po kolei zaczynając od góry drzewa
            node = self
            while node.value is None:
                #sprawdzenie czy element należy do lewej czy prawej strony podziału i przejście do odpowiedniego węzła
                if data[i, node.feature] <= node.threshold:
                    node = node.left
                else:
                    node = node.right
            y_pred[i] = node.value # Przypisanie przewidywanej klasy z liścia do wyniku
        return y_pred

In [ ]:
# Funkcja do znalezienia najlepszej głębokości drzewa za pomocą cross-validation, sprawdzana jest iteracyjnie accuracy dla glebokosi i wybrana ta z najlepsza wartoscia 
def find_best_depth(data: np.ndarray, target: np.ndarray,
                        depth_range=range(1, 20), k_folds=5, random_state=42) -> int:
        
        rng = np.random.default_rng(random_state)
        
        # Tworzenie losowych ustawień indeksów
        indices = np.arange(len(target))
        rng.shuffle(indices)
        
        # Dzielimy dane na k równych części (foldów)
        folds = np.array_split(indices, k_folds)
        
        best_depth = depth_range[0]
        best_score = -1.0

        for depth in depth_range:
            fold_scores = []
            # Sprawdzanie z pomocą cross-validation dla kolejnych głębokości drzewa
            for i in range(k_folds):
                
                val_idx = folds[i]
                train_idx = np.concatenate([folds[j] for j in range(k_folds) if j != i])

                X_train, X_val = data[train_idx], data[val_idx]
                y_train, y_val = target[train_idx], target[val_idx]

                # Trening drzewa z daną głębokością
                tree = TreeNode(max_depth=depth)
                tree.fit(X_train, y_train)

                # Liczenie celności na zbiorze walidacyjnym
                preds = tree.predict(X_val)
                accuracy = np.mean(preds == y_val)
                fold_scores.append(accuracy)

            # Średnia celności po wszystkich foldach dla danej głębokości
            mean_score = np.mean(fold_scores)

            if mean_score > best_score:
                best_score = mean_score
                best_depth = depth

        return best_depth

## Przykład trenowanie i testowania drzewa
 
Później znajduje się przykład trenowania i testowania drzewa na zbiorze danych `iris`, który zawierający 150 próbek irysów, z czego każda próbka zawiera 4 atrybuty: długość i szerokość płatków oraz długość i szerokość działki kielicha. Każda próbka należy do jednej z trzech klas: `setosa`, `versicolor` lub `virginica`, które są zakodowane jak int.

Możesz go wykorzystać do testowania swojej implementacji. Możesz też zaimplementować własne testy lub użyć innych zbiorów danych, np. innych [zbiorów danych z scikit-learn](https://scikit-learn.org/stable/datasets/toy_dataset.html#toy-datasets).

In [5]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

data = load_iris()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.33, random_state=2024)

tree_model = TreeNode()
tree_model.fit(X_train, y_train)
y_pred = tree_model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.84


In [ ]:
#kod do wywołania funkcji i treningu drzewa z doborem optymalnej głębokości drzewa
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


data = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.33, random_state=2024
)

# Znalezienie optymalnej głębokości
best_depth = find_best_depth(X_train, y_train, depth_range=range(3, 15))
print(f"optymalna głębokość: {best_depth}")

#Trening drzewa z dobraną głębokością
tree_model = TreeNode(max_depth=best_depth)
tree_model.fit(X_train, y_train)

y_pred = tree_model.predict(X_test)
print(accuracy_score(y_test, y_pred))

optymalna głębokość: 3
0.86
